# 11 - Behavioral Feature Extraction Pilot

Controlled TRAIN-only pilot step: extract behavioral BGP features from the 661 incident-specific 5-minute analytical windows.

This notebook does not create Normal labels, add CAIDA/RPKI enrichments, train models, run ablation, alter the frozen roster, or access Validation/Holdout.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

features = pd.read_parquet(PILOT / 'pilot_behavioral_features_v1.parquet')
inventory = pd.read_csv(PILOT / 'behavioral_feature_inventory_v1.csv')
quality = pd.read_csv(PILOT / 'behavioral_feature_quality_audit_v1.csv')
examples = pd.read_csv(PILOT / 'behavioral_feature_calculation_examples_v1.csv')
groups = pd.read_csv(PILOT / 'feature_group_summary_v1.csv')
windows = pd.read_csv(PILOT / 'five_minute_window_observation_counts_v1.csv')

## Representation Check

One incident-specific 5-minute window maps to one behavioral feature vector and one ML feature row. Raw BGP observations affect feature values only.

In [ ]:
print('windows:', len(windows))
print('feature rows:', len(features))
print('unique window ids:', features['window_id'].nunique())
print('all TRAIN:', set(features['partition']))
print('temporal statuses:', sorted(features['temporal_status'].unique()))

## Feature Inventory

In [ ]:
groups

In [ ]:
inventory[['feature_name', 'feature_group', 'source_fields', 'zero_window_behavior', 'verification_status']].head(20)

## Aggregation Examples

In [ ]:
examples

## Feature Table Preview

In [ ]:
features.head(10)

## Zero-Observation Windows

In [ ]:
zero = features[features['update_count'] == 0]
print('zero relevant-observation windows:', len(zero))
print('outage active zero relevant-observation windows:', len(zero[(zero['category'] == 'Outage') & (zero['temporal_status'] == 'INCIDENT_ACTIVE')]))
zero.groupby(['category', 'temporal_status']).size().reset_index(name='zero_windows')

## Feature Quality Audit

In [ ]:
quality['status'].value_counts()

In [ ]:
quality[quality['status'] != 'PASS']

## Guardrails

In [ ]:
assert len(features) == 661
assert features['window_id'].is_unique
assert set(features['partition']) == {'train'}
assert 'NORMAL' not in set(features['temporal_status'])
assert len(features[features['update_count'] == 0]) == 175
assert len(features[(features['category'] == 'Outage') & (features['temporal_status'] == 'INCIDENT_ACTIVE') & (features['update_count'] == 0)]) == 72
print('PASS: behavioral feature extraction pilot is window-level and TRAIN-only.')